<a href="https://colab.research.google.com/github/paurnavi25/Mini-RAG-Question-Answerer/blob/main/Mini_MultiPDF_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
os.listdir()

['.config',
 'BCS515B-module-1-pdf.pdf',
 'BCS515B-module-2-pdf.pdf',
 'BCS515B-module-3-pdf.pdf',
 'sample_data']

In [3]:
!pip install -q pypdf sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.2 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader
import os

documents = {}

for filename in os.listdir():
    if filename.endswith(".pdf"):
        reader = PdfReader(filename)

        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        documents[filename] = text

print("PDFs loaded:", list(documents.keys()))

PDFs loaded: ['BCS515B-module-1-pdf.pdf', 'BCS515B-module-2-pdf.pdf', 'BCS515B-module-3-pdf.pdf']


In [5]:
for filename, text in documents.items():
    print(filename, "→", len(text), "characters")

BCS515B-module-1-pdf.pdf → 43191 characters
BCS515B-module-2-pdf.pdf → 30630 characters
BCS515B-module-3-pdf.pdf → 17859 characters


In [6]:
for filename, text in documents.items():
    print("\n" + "="*60)
    print(filename)
    print("="*60)
    print(text[:500])


BCS515B-module-1-pdf.pdf
Artificial Intelligence-BCS515B 
 
 
 
Module 1 
 
 Artificial intelligence (AI) is a term to describe a branch of computer  science that is 
dedicated to creating intelligent machines that would learn to work and react like 
humans. 
 AI which stands for artificial intelligence refers to systems or machines that mimic 
human intelligence to perform tasks and can iteratively improve themselves based on 
the information they collect. 
 
1.1 WHAT IS AI? 
 
 The field of artificial intelligenc e

BCS515B-module-2-pdf.pdf
k 
1 
  
 
 
 
Artificial Intelligence   
 
Module 2 
PROBLEM-SOLVING 
 
 
 
1.  PROBLEM-SOLVING AGENTS 
Goal formulation, based on the current situation and the agent's performance measure, is the first step in 
problem solving. 
Problem formulation is the process of deciding what actions and states to consider, given a goal. 
An agent with several immediate options of unknown value can decide what to do by first examining future 
actions tha

In [7]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [8]:
all_chunks = []

for filename, text in documents.items():
    chunks = create_chunks(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })

print("Total chunks:", len(all_chunks))

Total chunks: 230


In [9]:
for i in range(3):
    print("\n--- CHUNK", i, "---")
    print("Source:", all_chunks[i]["source"])
    print(all_chunks[i]["text"][:500])


--- CHUNK 0 ---
Source: BCS515B-module-1-pdf.pdf
Artificial Intelligence-BCS515B 
 
 
 
Module 1 
 
 Artificial intelligence (AI) is a term to describe a branch of computer  science that is 
dedicated to creating intelligent machines that would learn to work and react like 
humans. 
 AI which stands for artificial intelligence refers to systems or machines that mimic 
human intelligence to perform tasks and can iteratively improve themselves based on 
the information they collect. 
 
1.1 WHAT IS AI? 
 
 The field of artificial intelligenc e

--- CHUNK 1 ---
Source: BCS515B-module-1-pdf.pdf
ased on 
the information they collect. 
 
1.1 WHAT IS AI? 
 
 The field of artificial intelligenc e, or AI, it attempts not just to understand but also to build 
intelligent entities. 
 AI currently encompasses a huge variety of subfields, ranging from the general (learning and 
perception) to the specific, such as playing chess, proving mathematical theorems, writing poetry, 
driving a car on 

In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
chunk_texts = [chunk["text"] for chunk in all_chunks]

In [13]:
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [14]:
print(embeddings.shape)

(230, 384)


In [15]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 230


In [16]:
question = "What is gradient descent?"

In [17]:
question_embedding = embedding_model.encode(
    [question]
)

question_embedding = np.array(question_embedding).astype("float32")

In [18]:
k = 3

distances, indices = index.search(
    question_embedding,
    k
)

In [19]:
for i, idx in enumerate(indices[0]):
    print("\n" + "="*60)
    print("Result:", i + 1)
    print("Source:", all_chunks[idx]["source"])
    print("Distance:", distances[0][i])
    print(all_chunks[idx]["text"][:1000])


Result: 1
Source: BCS515B-module-2-pdf.pdf
Distance: 1.3036702
nd.This will occur when the depth limit reaches 
d, the depth of the shallowest goal node. 
The algorithm is shown in Figure 3.18. Iterative deepening combines the benefits of depth-first and breadth- 
first search. 
 Like depth-first search, its memory requirements are modest: O(bd)to be precise.
 Like breadth-first search, it is complete when the branching factor is finite andoptimal when the path 
cost is a non decreasing function of the depth of the node.
 
  BCS515B
 s

Artificial Intel

Result: 2
Source: BCS515B-module-1-pdf.pdf
Distance: 1.3053235
 agent. 

Artificial Intelligence-BCS515B 
 
 
 
 The critic tells the learning element how well the agent is doing with respect to a fixed 
performance standard. The critic is necessary because the percepts themselves provide no 
indication of the agent’s success. 
 The last component of the learning agent is the problem generator . It is responsible for 
suggesting